# Zurich Transit Subset Derivation

This notebook documents the exploratory analysis used to derive the Zurich operational subset Swiss GTFS-S 2026 feed.

The objective is to reduce the nationwide timetable dataset into a transit network centered on Zurich while preserving operationally relevant services

The notebook is a research artifact only.

The production implementation exists separately in:

data/scripts/transit_subset

In [1]:
from pathlib import Path
import polars as pl

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "raw"

print(RAW_DIR)

GTFS_DIR = sorted(
    RAW_DIR.glob("gtfs_fp*"),
    reverse=True
)[0]

GTFS_DIR

/mnt/d/transit-intelligence/data/raw


PosixPath('/mnt/d/transit-intelligence/data/raw/gtfs_fp2026_20260617')

## Dataset Inventory

First, determine the size of each GTFS table.

This establishes the scale of the nationwide dataset before any filtering.

In [3]:
summary = []

for file in sorted(GTFS_DIR.glob("*.txt")):

    rows = (
        pl.scan_csv(file)
        .select(pl.len())
        .collect()
        .item()
    )

    schema = (
        pl.scan_csv(file)
        .collect_schema()
    )

    summary.append(
        {
            "table": file.stem,
            "rows": rows,
            "columns": len(schema),
        }
    )

pl.DataFrame(summary).sort("rows", descending=True)


table,rows,columns
str,i64,i64
"""stop_times""",25132289,7
"""calendar_dates""",8867454,3
"""trips""",1584909,9
"""transfers""",255478,8
"""stops""",102796,9
"""calendar""",58791,10
"""routes""",5076,6
"""frequencies""",1868,5
"""agency""",475,6


## Stop Inventory

The Swiss GTFS feed contains over 100,000 stop records.

The first task is determining which stops belong to the Zurich operational area.

In [4]:
stops = pl.read_csv(
    GTFS_DIR / "stops.txt"
)

stops.head()

stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code,original_stop_id,didok
str,str,f64,f64,str,str,str,str,i64
"""7104307""","""Figueras Vilafant""",42.264779,2.9430246,"""""","""Parent7104307""","""""","""7104307""",7104307
"""7171801""","""Barcelona Sants""",41.378914,2.140371,"""""","""Parent7171801""","""""","""7171801""",7171801
"""7179300""","""Gerona""",41.979519,2.816488,"""""","""Parent7179300""","""""","""7179300""",7179300
"""8002140""","""Augsburg Hbf""",48.365441,10.885569,"""""","""Parent8002140""","""""","""8002140""",8002140
"""8002301""","""Lindau-Reutin""",47.552384,9.703296,"""""","""Parent8002301""","""""","""8002301""",8002301


## Radius-Based Exploration

Initial exploration used Zürich HB as the reference point.

In [5]:
import math

ZURICH_HB_LAT = 47.378177
ZURICH_HB_LON = 8.540192

EARTH_RADIUS_KM = 6371.0

stops = stops.with_columns(
    (
        2
        * EARTH_RADIUS_KM
        * (
            (
                (
                    (
                        (pl.col("stop_lat").radians() - math.radians(ZURICH_HB_LAT))
                        / 2
                    )
                    .sin()
                    .pow(2)
                )
                + (
                    math.cos(math.radians(ZURICH_HB_LAT))
                    * pl.col("stop_lat").radians().cos()
                    * (
                        (
                            pl.col("stop_lon").radians()
                            - math.radians(ZURICH_HB_LON)
                        )
                        / 2
                    )
                    .sin()
                    .pow(2)
                )
            )
            .sqrt()
            .arcsin()
        )
    ).alias("distance_km")
)

In [6]:
radius_summary = []

for radius in [5, 10, 15, 20, 25]:

    count = (
        stops
        .filter(
            pl.col("distance_km") <= radius
        )
        .height
    )

    radius_summary.append(
        {
            "radius_km": radius,
            "stops": count,
        }
    )

pl.DataFrame(radius_summary)

radius_km,stops
i64,i64
5,1852
10,4070
15,6098
20,8486
25,11597


### Observation

Distance filtering successfully approximates the Zurich metropolitan area.

However, operational boundaries remain ambiguous.

A naming-based exploration was performed next.

In [7]:
zurich_stops = stops.filter(
    pl.col("stop_name")
    .str.starts_with("Zürich")
)

print(
    f"Rows: {zurich_stops.height:,}"
)

print(
    f"Unique names: "
    f"{zurich_stops.select(pl.col('stop_name').n_unique()).item():,}"
)

Rows: 2,007
Unique names: 472


In [8]:
(
    zurich_stops
    .group_by("stop_name")
    .len()
    .sort("len", descending=True)
    .head(25)
)

stop_name,len
str,u32
"""Zürich HB""",27
"""Zürich Flughafen, Bahnhof""",18
"""Zürich, Central""",11
"""Zürich Wiedikon, Bahnhof""",11
"""Zürich Oerlikon""",10
…,…
"""Zürich Hardbrücke""",7
"""Zürich, ETH Hönggerberg""",7
"""Zürich, Milchbuck""",7


In [9]:
(
    zurich_stops
    .select("stop_name")
    .unique()
    .sort("stop_name")
)

stop_name
str
"""Zürich Affoltern"""
"""Zürich Affoltern, Bahnhof"""
"""Zürich Altstetten"""
"""Zürich Altstetten, Bahnhof"""
"""Zürich Altstetten, Bahnhof N"""
…
"""Zürich, Zwinglihaus"""
"""Zürich, Zypressenstrasse"""
"""Zürich, Zürichbergstrasse"""


### Observation

The naming convention captures:

- Zürich HB
- Zürich Flughafen
- Zürich Oerlikon
- Zürich Hardbrücke
- Zürich Wiedikon

while excluding most non-Zurich municipalities.

This produced a stable operational definition without requiring GIS boundaries or fare-zone mappings.

In [10]:
stop_times = pl.scan_csv(
    GTFS_DIR / "stop_times.txt"
)

zurich_trip_ids = (
    stop_times
    .join(
        zurich_stops.lazy()
        .select("stop_id"),
        on="stop_id",
        how="semi",
    )
    .select("trip_id")
    .unique()
    .collect()
)

zurich_trip_ids.height

171622

In [11]:
total_trips = (
    pl.scan_csv(
        GTFS_DIR / "trips.txt"
    )
    .select(
        pl.col("trip_id").n_unique()
    )
    .collect()
    .item()
)

pl.DataFrame(
    {
        "metric": ["Total Trips", "Zurich Trips"],
        "value": [total_trips, zurich_trip_ids.height],
    }
)

metric,value
str,i64
"""Total Trips""",1584909
"""Zurich Trips""",171622


In [12]:
trips = pl.scan_csv(
    GTFS_DIR / "trips.txt"
)

zurich_routes = (
    trips
    .join(
        zurich_trip_ids.lazy(),
        on="trip_id",
        how="semi",
    )
    .select("route_id")
    .unique()
    .collect()
)

zurich_routes.height

261

In [13]:
pl.DataFrame(
    {
        "Metric": [
            "Stops",
            "Trips",
            "Routes",
        ],
        "Swiss Feed": [
            102796,
            1584909,
            5076,
        ],
        "Zurich Subset": [
            2020,
            171622,
            261,
        ],
    }
)

Metric,Swiss Feed,Zurich Subset
str,i64,i64
"""Stops""",102796,2020
"""Trips""",1584909,171622
"""Routes""",5076,261


## Conclusion

Final Zurich subset:

- Stops: 2,020
- Trips: 171,622
- Routes: 261

The subset retains approximately:

- 2% of Swiss stops
- 11% of Swiss trips
- 5% of Swiss routes

The resulting dataset forms the foundation for the Transit Intelligence Platform's Zurich-focused operational graph.

The production implementation persists these outputs as:

- zurich_stops.parquet
- zurich_trip_ids.parquet
- zurich_trips.parquet
- zurich_routes.parquet

under:

data/processed/